In [1]:
import os
import torch
import torchvision.transforms as transforms
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
import timm
from torchvision.transforms import functional as F
import qoi
from PIL import Image
from tqdm import tqdm
import numpy as np


os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"
# QOI 캐시
qoi_cache = {}

def mixup_data(x, y, alpha=1.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def qoi_loader(qoi_path):
    if qoi_path in qoi_cache:
        return qoi_cache[qoi_path]

    if not os.path.exists(qoi_path):
        print(f"🚨 파일 없음: {qoi_path}")
        return None

    with open(qoi_path, "rb") as f:
        qoi_data = f.read()

    img = qoi.decode(qoi_data)
    img = Image.fromarray(img).convert("RGB")
    qoi_cache[qoi_path] = img
    return img

class ResizeWithPadding:
    def __init__(self, size=224, padding_color=(0, 0, 0)):
        self.size = size
        self.padding_color = padding_color

    def __call__(self, img):
        w, h = img.size
        scale = self.size / max(w, h)
        new_w, new_h = int(w * scale), int(h * scale)
        img = F.resize(img, (new_h, new_w))

        delta_w = self.size - new_w
        delta_h = self.size - new_h
        padding = (delta_w // 2, delta_h // 2, delta_w - delta_w // 2, delta_h - delta_h // 2)

        return F.pad(img, padding, fill=self.padding_color, padding_mode="constant")

transform_train = transforms.Compose([
    ResizeWithPadding(size=224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_test = transforms.Compose([
    ResizeWithPadding(size=224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_path = "/root/Public_Storage/madelab_khw/lpcv/coco/cropped_dir_train"
test_path = "/root/Public_Storage/madelab_khw/lpcv/coco/cropped_dir_test"

dataset = ImageFolder(root=train_path, transform=transform_train, loader=qoi_loader, is_valid_file=lambda path: path.endswith(".qoi"))
test_dataset = ImageFolder(root=test_path, transform=transform_test, loader=qoi_loader, is_valid_file=lambda path: path.endswith(".qoi"))

train_size = int(0.8 * len(dataset))
valid_size = len(dataset) - train_size
train_dataset, valid_dataset = random_split(dataset, [train_size, valid_size])

batch_size = 128
num_workers = 0

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

# ViT-B 모델 불러오기
num_classes = 64
model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.device_count() > 1:
    print(f"🔹 {torch.cuda.device_count()} 개의 GPU 사용 중")
    model = torch.nn.DataParallel(model)

model = model.to(device)
torch.backends.cudnn.benchmark = True

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

num_epochs = 10
best_valid_acc = 0.0
save_path = "/root/Public_Storage/madelab_khw/lpcv/model/vit_base_patch16_224.pth"

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")
    mixup_prob = 0.5

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)

        if np.random.rand() < mixup_prob:
            images, targets_a, targets_b, lam = mixup_data(images, labels, alpha=0.4)
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)

            _, predicted = outputs.max(1)
            correct += lam * predicted.eq(targets_a).sum().item() + (1 - lam) * predicted.eq(targets_b).sum().item()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        total += labels.size(0)
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_acc = 100 * correct / total
    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    valid_acc = 100 * correct / total

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        torch.save(model.module.state_dict() if torch.cuda.device_count() > 1 else model.state_dict(), save_path)

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss:.4f}, Train Acc: {train_acc:.2f}%, Valid Acc: {valid_acc:.2f}%")

print("✅ 학습 완료!")


🔹 2 개의 GPU 사용 중


Epoch 1/10:   0%|                                                                                           | 0/1433 [00:00<?, ?batch/s]

Epoch 1/10:   0%|                                                                              | 0/1433 [00:02<?, ?batch/s, loss=4.6157]

Epoch 1/10:   0%|                                                                    | 1/1433 [00:02<1:06:41,  2.79s/batch, loss=4.6157]

Epoch 1/10:   0%|                                                                    | 1/1433 [00:04<1:06:41,  2.79s/batch, loss=4.2605]

...

Epoch 6/10:  45%|██████████████████████████████▉                                     | 651/1433 [14:02<16:19,  1.25s/batch, loss=0.7624]

Epoch 6/10:  45%|██████████████████████████████▉                                     | 652/1433 [14:02<16:17,  1.25s/batch, loss=0.7624]

Epoch 6/10:  45%|██████████████████████████████▉                                     | 652/1433 [14:03<16:17,  1.25s/batch, loss=0.9357]

Epoch 6/10:  46%|██████████████████████████████▉                                     | 653/1433 [14:03<16:16,  1.25s/batch, loss=0.9357]

Epoch 6/10:  46%|██████████████████████████████▉                                     | 653/1433 [14:04<16:16,  1.25s/batch, loss=1.0580]